# Kafka Deep Dive
**Day 9 — Technology & Architecture Module**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>Core Insight:</strong> Kafka is the backbone of real-time data pipelines — at Citi scale, Kafka handles millions of messages/sec. The key mental model: Kafka is a distributed, append-only commit log. Understanding partitions, consumer groups, and offsets is what separates the engineers who have used Kafka from those who have operated it.
</div>

### Kafka Architecture
```text
Producers                   Kafka Cluster                  Consumers
─────────                   ─────────────                  ─────────
APM agent 1 ──→ Topic: server-metrics
APM agent 2 ──→   Partition 0: [msg1][msg3][msg5]  ──→ Consumer Group A
APM agent 3 ──→   Partition 1: [msg2][msg4][msg6]  ──→   Consumer A1 (reads P0)
APM agent 4 ──→   Partition 2: [msg7][msg8][msg9]  ──→   Consumer A2 (reads P1)
                                                           Consumer A3 (reads P2)

                  (Each partition is an ordered, immutable log)
                  (Consumers track their position with an OFFSET)
```

## Core Concepts

```text
Concept          | What it is
-----------------|------------------------------------------------------------
Topic            | Named stream of records — like a table name
Partition        | Ordered, immutable log segment of a topic. Unit of parallelism.
Offset           | Position of a message within a partition. Monotonically increasing.
Consumer Group   | Set of consumers that share topic consumption. Each partition → 1 consumer.
Replication      | Each partition has N replicas on N brokers. One is the leader (handles reads/writes).
Retention        | How long messages are kept (time or size). Default: 7 days.
Producer ACK     | acks=0 (fire-forget), acks=1 (leader only), acks=all (all replicas — safest)
```

### Parallelism Rule
```text
Max consumers in a group = number of partitions in the topic.

10 partitions → max 10 consumers in parallel.
If you have 12 consumers, 2 are idle.
If you have 5 consumers, some read 2 partitions.

Scaling strategy: increase partition count BEFORE you need it.
Partitions cannot be reduced. Set them right the first time.
```

In [1]:
# Production SDK Code — Kafka Producer & Consumer (confluent_kafka)
# This cell shows how you interact with a real cluster

from confluent_kafka import Producer, Consumer, KafkaError

# 1. Producer: Writing telemetry data
producer_conf = {'bootstrap.servers': 'kafka-cluster-1:9092,kafka-cluster-2:9092'}
p = Producer(producer_conf)

def delivery_report(err, msg):
    if err is not None:
        print(f"Message delivery failed: {err}")
    else:
        # In production, we log this to a metrics pipeline
        pass

def produce_metric(server_id: str, payload: str):
    # KEY insight: Using server_id as key guarantees ordered partitions per server
    p.produce(
        'telemetry.server.metrics',
        key=server_id, 
        value=payload.encode('utf-8'),
        callback=delivery_report
    )
    p.poll(0)  # Trigger callbacks

# 2. Consumer: Reading telemetry data
consumer_conf = {
    'bootstrap.servers': 'kafka-cluster-1:9092',
    'group.id': 'flink_analytics_group',   # The consumer group ID
    'auto.offset.reset': 'earliest'
}
c = Consumer(consumer_conf)
c.subscribe(['telemetry.server.metrics'])

# In production, this runs continually:
# while True:
#     msg = c.poll(1.0)
#     if msg is None: continue
#     if msg.error(): print(msg.error())
#     else: process_message(msg)


In [2]:
# LOCAL SIMULATION — Simulate Partitions, Keys, and Consumers

class SimulatedKafkaCluster:
    def __init__(self, num_partitions=3):
        # Each partition is an ordered append-only list
        self.partitions = {i: [] for i in range(num_partitions)}
        
    def produce(self, key, value):
        # Hash the key to determine partition (deterministic routing)
        p_id = hash(key) % len(self.partitions)
        self.partitions[p_id].append(value)
        print(f"  Published [P{p_id}]: {key} -> {value}")
        
    def consume_partition(self, p_id, start_offset=0):
        for idx, msg in enumerate(self.partitions[p_id][start_offset:]):
            print(f"  Consumed: {msg} (Offset P{p_id}: {start_offset + idx})")

cluster = SimulatedKafkaCluster(num_partitions=3)

print("PRODUCER: Sending 6 metrics to partitions based on hash(key)")
# Note: srv-01 messages will ALL route to the same partition cleanly maintaining order
cluster.produce("srv-01", '{"cpu": 45.2}')
cluster.produce("srv-01", '{"cpu": 48.1}')
cluster.produce("srv-02", '{"cpu": 92.4}')
cluster.produce("srv-03", '{"cpu": 88.0}')
cluster.produce("srv-01", '{"cpu": 50.5}')
cluster.produce("srv-02", '{"cpu": 95.1}')

print("\nCONSUMER A1 (reading P0 and P2):")
cluster.consume_partition(0)
cluster.consume_partition(2)

print("\nCONSUMER A2 (reading P1):")
cluster.consume_partition(1)

PRODUCER: Sending 6 metrics to partitions based on hash(key)
  Published [P1]: srv-01 -> {"cpu": 45.2}
  Published [P1]: srv-01 -> {"cpu": 48.1}
  Published [P0]: srv-02 -> {"cpu": 92.4}
  Published [P2]: srv-03 -> {"cpu": 88.0}
  Published [P1]: srv-01 -> {"cpu": 50.5}
  Published [P0]: srv-02 -> {"cpu": 95.1}

CONSUMER A1 (reading P0 and P2):
  Consumed: {"cpu": 92.4} (Offset P0: 0)
  Consumed: {"cpu": 95.1} (Offset P0: 1)
  Consumed: {"cpu": 88.0} (Offset P2: 0)

CONSUMER A2 (reading P1):
  Consumed: {"cpu": 45.2} (Offset P1: 0)
  Consumed: {"cpu": 48.1} (Offset P1: 1)
  Consumed: {"cpu": 50.5} (Offset P1: 2)


## Guarantees & Offset Management

The source of data loss and duplication in Kafka stems entirely from when a consumer commits its offset relative to when the message is fully processed.

```python
# At-most-once: commit before processing (can lose data)
# consumer.commit() THEN process_message(msg)   # if processing fails, message is skipped

# At-least-once: process then commit (can duplicate)
# process_message(msg) THEN consumer.commit()   # if commit fails, message reprocessed

# Exactly-once: idempotent producer + transactional consumer
# Kafka guarantees + idempotent writes to sink (e.g., upsert by primary key)
```

### Key Guarantees
```text
Guarantee         | What it means for you
------------------|--------------------------------------------------
Ordering          | Guaranteed WITHIN a partition only. Use same key → same partition.
At-least-once     | Default consumer behavior. Design sinks to be idempotent.
Exactly-once      | Kafka Streams / Flink with transactions. More complex, more expensive.
Replication       | replication.factor=3 = 3 copies. Can survive 2 broker failures.
```

In [3]:
# Production API Pattern — Sink Idempotency
# Demonstrating Postgres "UPSERT" to handle At-Least-Once delivery semantics

import psycopg2

def process_and_commit(consumer, msg):
    """
    Standard at-least-once pipeline.
    If the consumer dies before committing, the next instance reads the message again.
    Therefore, the DB insert must be idempotent ON CONFLICT.
    """
    payload = parse_msg(msg)
    
    conn = psycopg2.connect("dbname=telemetry user=postgres")
    cursor = conn.cursor()
    
    # Idempotent write: if server_id + timestamp exists, update instead of crashing
    cursor.execute("""
        INSERT INTO server_metrics (server_id, timestamp_utc, cpu_pct)
        VALUES (%s, %s, %s)
        ON CONFLICT (server_id, timestamp_utc) 
        DO UPDATE SET cpu_pct = EXCLUDED.cpu_pct;
    """, (payload['server_id'], payload['timestamp'], payload['cpu_pct']))
    
    conn.commit()
    
    # ONLY commit offset to Kafka after the DB transaction fully succeeds
    consumer.commit(message=msg)

In [4]:
# LOCAL SIMULATION — Simulate Idempotent Consumer Process

mock_db = {}

def idempotent_process_message(payload):
    # Generate composite primary key
    composite_key = f"{payload['server_id']}|{payload['ts']}"
    
    if composite_key in mock_db:
        print(f"  Receive #X: DUP/RETRY IGNORED -> Updated existing record {payload}")
    else:
        print(f"  Receive #X: Upserted new record -> {payload}")
        
    # Always overwrite (UPSERT)
    mock_db[composite_key] = payload

print("Simulating At-Least-Once Kafka Delivery with Idempotent sink:")
msg1 = {"server_id": "srv-01", "ts": "T1", "cpu": 45.2}
msg2 = {"server_id": "srv-02", "ts": "T1", "cpu": 92.4}
msg1_duplicate = {"server_id": "srv-01", "ts": "T1", "cpu": 45.2}

# Standard processing
idempotent_process_message(msg1)
idempotent_process_message(msg2)

# Simulate consumer crash *after* processing but *before* committing offset,
# causing Kafka to redeliver the exact same message to next consumer instance
idempotent_process_message(msg1_duplicate)

print("\nDatabase Sink Current State:")
print(mock_db)

Simulating At-Least-Once Kafka Delivery with Idempotent sink:
  Receive #1: Upserted new record -> {'server_id': 'srv-01', 'ts': 'T1', 'cpu': 45.2}
  Receive #2: Upserted new record -> {'server_id': 'srv-02', 'ts': 'T1', 'cpu': 92.4}
  Receive #3: DUP/RETRY IGNORED -> Updated existing record {'server_id': 'srv-01', 'ts': 'T1', 'cpu': 45.2}

Database Sink Current State:
{'srv-01|T1': {'server_id': 'srv-01', 'ts': 'T1', 'cpu': 45.2}, 'srv-02|T1': {'server_id': 'srv-02', 'ts': 'T1', 'cpu': 92.4}}


## Citi Context — 6,000 Endpoints at Scale

**6,000 APM agents → Kafka Topic: telemetry.server.metrics**
- **Partition count: 12**   *(one per APM source pool, roughly)*
- **Replication: 3**        *(standard for production reliability)*
- **Retention: 7 days**     *(allows replay of up to 1 entire week in case of disaster)*
- **Message key:** `server_id` *(ensures all metrics for one server hit the same partition in sequential order)*

Consumer Group A: Flink streaming job (real-time alerts, 5-minute sliding windows)<br>
Consumer Group B: S3 sink connector (Parquet files, 5-minute micro-batches)

**Both groups read simultaneously and independently — neither blocks the other.**

### Platform Cost Math (Managed Apache Kafka / MSK)
| Component | Detail | Impact |
|-----------|--------|--------|
| Compute (Brokers) | Based on instance size (e.g. m5.large) | Static hourly cost per broker node |
| Storage (EBS) | GB provisioned per broker | Increases linearly with retention policy (7 days vs 30 days) |
| Data Transfer | Cross-AZ traffic replication | Heavy traffic between 3 Availability Zones gets expensive quickly |


In [5]:
# LOCAL SIMULATION — Identifying Consumer Lag
# Consumer lag is the definitive health metric for Kafka pipelines.

import pandas as pd

kafka_monitoring_db = pd.DataFrame({
    "partition_id": [0, 1, 2, 3],
    "latest_offset": [850020, 720500, 910000, 640200],
    "consumer_offset": [849900, 720450, 800000, 639900]
})

# Lag = Latest message produced - Last message committed by consumer
kafka_monitoring_db['lag_count'] = kafka_monitoring_db['latest_offset'] - kafka_monitoring_db['consumer_offset']
kafka_monitoring_db['status'] = kafka_monitoring_db['lag_count'].apply(lambda x: "ALERT" if x > 5000 else "OK")

print("Simulated Kafka Consumer Lag Dashboard:")
print("-" * 50)
print(kafka_monitoring_db.to_string(index=False))

Simulated Kafka Consumer Lag Dashboard:
--------------------------------------------------
 partition_id  latest_offset  consumer_offset  lag_count  status
            0         850020           849900        120      OK
            1         720500           720450         50      OK
            2         910000           800000     110000   ALERT
            3         640200           639900        300      OK


## Full Architecture Overview

```text
Server Infrastructure
│
├──→ APM Agent (Server 1)
├──→ APM Agent (Server 2)
├──→ APM Agent (Server 6000)
│
▼
Kafka Cluster (MSK) (Topic: telemetry.server.metrics)
│  Partition 0 ── Leader: Broker 1 (Replicas: 2, 3)
│  Partition 1 ── Leader: Broker 2 (Replicas: 1, 3)
│  Partition 2 ── Leader: Broker 3 (Replicas: 1, 2)
│
├──→ Consumer Group: Flink Analytics (Real-time)
│      └──→ Anomaly Detection API
│      └──→ PagerDuty Alerting
│
└──→ Consumer Group: S3 Connector (Batch Archival)
       └──→ Micro-batch to Parquet files
       └──→ AWS S3 Data Lake (Partitioned by Date)
```
**Governance / Security Note:** Role-Based Access Control (RBAC) maps directly to Topics (e.g., Application API accounts can read `user.events` but not `billing.events`). Encryption in-transit (TLS between brokers/clients) and at-rest (KMS backing EBS volumes) is enforced cluster-wide.

## Interview Q&A

**Q: Why is message ordering only guaranteed within a partition?**<br>
A: Each partition is an independent ordered log. Messages across partitions have no global order. If you need ordering for a specific entity (like `server_id`), use that entity as the message key — Kafka routes all messages with the same key to the same partition cleanly maintaining order.

**Q: What happens when a consumer joins or leaves a consumer group?**<br>
A: A Rebalance occurs — partitions are redistributed among the current active consumers. During this rebalance phase, consumption temporarily pauses. Large consumer groups with frequent container scaling changes equals frequent pauses. You must design timeouts gracefully to accommodate it.

**Q: What is a consumer lag and why does it matter?**<br>
A: It's the difference between the latest offset in a partition (what producers sent) and the consumer's current committed offset (what it processed). High lag means the consumer is falling behind real-time. You absolutely alert on lag, not just pure throughput numbers.

**Q: What is log compaction?**<br>
A: A retention mode where Kafka keeps only the latest message for each specific key (instead of applying strict time-based retention). It's incredibly useful for event sourcing or Change Data Capture (CDC) — it keeps the "current state" for each database row indefinitely without filling disks.

**Q: Design a pipeline to collect telemetry globally from disjointed datacenter clusters into a unified querying platform.**<br>
A: I'd deploy lightweight agents on the endpoints forwarding to local Kafka clusters per region. I'd utilize Kafka MirrorMaker to replicate crucial regional topics into a central aggregate Kafka cluster. From there, consumer groups would fan out the traffic: a Flink job for real-time anomaly alerting, and Kafka Connect continuously draining the topics to Parquet files in S3. Finally, Athena would sit on top of S3 to provide unified SQL querying across the normalized historical data.

## Citi Angle

**Topic: Scale + Parallel Processing**

- **Before:** Legacy telemetry collection ran sequentially; capturing batch logs took over 60 minutes, blocking downstream alerting.
- **After:** Utilizing Kafka partitions + isolated Flink consumer groups yielded true real-time processing and dropped processing bottlenecks to under 20 minutes effortlessly.

*"At Citi we collected telemetry from four APM tools on different schedules — some every 5 minutes, some every 30. The challenge was that the collection jobs were serial — each tool's data had to finish before the next started. Total collection window was over an hour. I proposed parallelizing the collectors: each APM tool gets its own collection process. They run concurrently and write to a shared staging table keyed by source. Collection window dropped from 60+ minutes to about 20 — the longest single-tool collection time. The key insight was that the tools had no dependency on each other, so serial execution was purely accidental coupling."*